<a href="https://colab.research.google.com/github/VictoriaO26/AI-assisted-Cloud-Infrastructure-Cybersecurity-Bootcamp/blob/main/Module%204/Module_4_Malware_Classification_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 4: Malware Image Classification CNN Lab





**Objective**: The students will train a malware family classifier using behavioral API-call data to identify malware types or families.

**Acknowledgement**: Acknowledgment: This module was developed under the National Science Foundation awards # 2416990, 2416992, and 2607393 at Tennessee Tech University, North Carolina A&T State University, and the University at Albany.








This Google Colab notebook classifies inert malware bytecode images into malware families with a convolutional neural network (CNN). It does not execute malware.

The dataset is downloaded from the `module_4` directory of the `acr-research-lab/AI4SecureCI` GitHub repository using sparse checkout:

- `train400/` — required training data
- `studentTest200/` — required independent test data
- `anomalies/` — optional supporting folder; it is not used by the core CNN pipeline

The trained compatibility model is saved to:

```text
/content/malware_cnn_model.h5
```

Google Drive is optional and is used only when `SAVE_MODEL_TO_DRIVE = True`.




## Student Notebook Guidance

Run the notebook from top to bottom after completing TODOs 1–12. Use the outputs generated by your own run to answer the post-assignment questions in the student guide.


## Expected project structure

Source location:

```text
https://github.com/acr-research-lab/AI4SecureCI/tree/main/module_4
```

Expected repository structure:

```text
module_4/
├── train400/          # required
├── studentTest200/    # required
└── anomalies/         # optional for this CNN lab
```

After the download cell runs, Colab uses:

```text
/content/
├── train400/
├── studentTest200/
├── anomalies/         # present only when available
└── malware_cnn_model.h5
```


## 1. Runtime setup

1. Open the notebook in Google Colab.
2. Select **Runtime → Change runtime type → GPU** when available.
3. Run the notebook from top to bottom after completing the required TODOs.

A GPU is recommended but is not required for correctness.


In [ ]:
import os
import time
import shutil
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

print("GPU devices:", tf.config.list_physical_devices("GPU"))


## 2. Configure the GitHub source and local paths

The required dataset folders are pulled from the `module_4` directory of the repository using sparse checkout. No dataset ZIP file is used.


In [ ]:
GITHUB_REPO_URL = "https://github.com/acr-research-lab/AI4SecureCI.git"
GITHUB_BRANCH = "main"
GITHUB_BASE_PATH = "module_4"

PROJECT_DIR = Path("/content")
TMP_DIR = PROJECT_DIR / "ai4secureci_module4_tmp"
TRAIN_DIR = PROJECT_DIR / "train400"
TEST_DIR = PROJECT_DIR / "studentTest200"
ANOMALIES_DIR = PROJECT_DIR / "anomalies"
MODEL_PATH = PROJECT_DIR / "malware_cnn_model.h5"

# Google Drive is optional. Change this to True only when you want the model
# copied to MyDrive after training.
SAVE_MODEL_TO_DRIVE = False
DRIVE_MODEL_PATH = Path("/content/drive/MyDrive/malware_cnn_model.h5")

# Make the path values available to the Bash download cell.
os.environ["GITHUB_REPO_URL"] = GITHUB_REPO_URL
os.environ["GITHUB_BRANCH"] = GITHUB_BRANCH
os.environ["GITHUB_BASE_PATH"] = GITHUB_BASE_PATH
os.environ["PROJECT_DIR"] = str(PROJECT_DIR)
os.environ["TMP_DIR"] = str(TMP_DIR)
os.environ["TRAIN_DIR"] = str(TRAIN_DIR)
os.environ["TEST_DIR"] = str(TEST_DIR)
os.environ["ANOMALIES_DIR"] = str(ANOMALIES_DIR)

print("GitHub repository:", GITHUB_REPO_URL)
print("GitHub branch:", GITHUB_BRANCH)
print("GitHub base path:", GITHUB_BASE_PATH)
print("Training path:", TRAIN_DIR)
print("Independent test path:", TEST_DIR)
print("Optional anomalies path:", ANOMALIES_DIR)
print("Model path:", MODEL_PATH)
print("Save model to Drive:", SAVE_MODEL_TO_DRIVE)


## 3. Optional Google Drive mounting

The dataset does not come from Google Drive. Drive is mounted only when `SAVE_MODEL_TO_DRIVE = True` so that the trained model can be copied to `MyDrive`.


In [ ]:
if SAVE_MODEL_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=True)
    print("Google Drive mounted for model saving.")
else:
    print("Google Drive is not required. The model will remain in /content.")


## 4. Download the dataset folders into `/content`

This cell downloads the two required folders and attempts to download the optional `anomalies/` folder. A missing optional folder does not stop the core lab.


In [ ]:
%%bash
set -euo pipefail

REQUIRED_FOLDERS=("train400" "studentTest200")
OPTIONAL_FOLDERS=("anomalies")

cd "$PROJECT_DIR"

if ! git lfs version >/dev/null 2>&1; then
  apt-get update -qq
  apt-get install -y -qq git-lfs
fi

# Initialize LFS without printing the harmless initialization message.
git lfs install --skip-repo >/dev/null 2>&1

rm -rf "$TMP_DIR" "$TRAIN_DIR" "$TEST_DIR" "$ANOMALIES_DIR"

# Do not automatically download LFS objects during cloning.
GIT_LFS_SKIP_SMUDGE=1 git clone \
  --quiet \
  --depth 1 \
  --filter=blob:none \
  --sparse \
  --branch "$GITHUB_BRANCH" \
  "$GITHUB_REPO_URL" \
  "$TMP_DIR"

cd "$TMP_DIR"

# Also prevent automatic LFS downloads during sparse checkout.
GIT_LFS_SKIP_SMUDGE=1 git sparse-checkout set \
  "$GITHUB_BASE_PATH/train400" \
  "$GITHUB_BASE_PATH/studentTest200" \
  "$GITHUB_BASE_PATH/anomalies"

# Download required LFS files separately.
# Any missing required object should remain a real, visible error.
for folder in "${REQUIRED_FOLDERS[@]}"; do
  repo_path="$GITHUB_BASE_PATH/$folder"

  if ! git lfs pull \
      --include="$repo_path/**" \
      --exclude=""; then
    echo "ERROR: Git LFS could not download required folder: $folder"
    exit 1
  fi
done

# Move required folders.
for folder in "${REQUIRED_FOLDERS[@]}"; do
  src="$TMP_DIR/$GITHUB_BASE_PATH/$folder"
  dest="$PROJECT_DIR/$folder"

  if [ ! -d "$src" ]; then
    echo "ERROR: Required folder was not downloaded: $src"
    exit 1
  fi

  mv "$src" "$dest"
done

# Attempt optional folders without displaying Git LFS scanner errors.
for folder in "${OPTIONAL_FOLDERS[@]}"; do
  repo_path="$GITHUB_BASE_PATH/$folder"
  src="$TMP_DIR/$repo_path"
  dest="$PROJECT_DIR/$folder"
  lfs_log="$TMP_DIR/git-lfs-${folder}.log"

  if git lfs pull \
      --include="$repo_path/**" \
      --exclude="" \
      >"$lfs_log" 2>&1; then

    if [ -d "$src" ]; then
      mv "$src" "$dest"
      echo "Optional folder downloaded: $dest"
    else
      echo "Optional folder not found; continuing without it: $folder"
    fi
  else
    # Remove any incomplete folder containing unresolved LFS pointers.
    rm -rf "$src"
    echo "Optional folder could not be downloaded; continuing without it: $folder"
  fi
done

rm -rf "$TMP_DIR"

echo
echo "Required folders are ready:"
printf ' - %s\n' "$TRAIN_DIR" "$TEST_DIR"

if [ -d "$ANOMALIES_DIR" ]; then
  printf ' - %s (optional)\n' "$ANOMALIES_DIR"
fi

## 5. Verify the dataset folders

`train400/` and `studentTest200/` must contain matching class-subfolder names. The notebook prints image counts so that class balance can be discussed in the post-assignment questions.


In [ ]:
def summarize_image_folder(root: Path) -> dict[str, int]:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"Missing required folder: {root}")

    class_dirs = sorted(
        p for p in root.iterdir()
        if p.is_dir() and not p.name.startswith(".")
    )
    if not class_dirs:
        raise ValueError(f"No class subfolders found inside: {root}")

    image_exts = {".png", ".jpg", ".jpeg", ".bmp", ".gif"}
    counts = {
        class_dir.name: sum(
            1 for item in class_dir.rglob("*")
            if item.is_file() and item.suffix.lower() in image_exts
        )
        for class_dir in class_dirs
    }

    print(f"\n{root}")
    print(f"Classes: {len(counts)}")
    print(f"Images:  {sum(counts.values())}")
    for name, count in counts.items():
        print(f"  {name}: {count}")

    return counts


train_counts = summarize_image_folder(TRAIN_DIR)
test_counts = summarize_image_folder(TEST_DIR)

if list(train_counts) != list(test_counts):
    raise ValueError(
        "train400 and studentTest200 must contain the same class folders "
        "in the same sorted order."
    )

if ANOMALIES_DIR.exists():
    print(f"\nOptional anomalies folder is available: {ANOMALIES_DIR}")
else:
    print("\nOptional anomalies folder is not available. The core CNN lab can continue.")


## 6. Load the training, validation, and independent test datasets

The notebook splits `train400/` into training and validation subsets. `studentTest200/` is loaded with `shuffle=False` and is reserved for evaluation after training.


In [ ]:
IMG_SIZE = (64, 64)
BATCH_SIZE = 32
VAL_SPLIT = 0.10
SEED = 123

tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()

# TODO 1: Load the training subset from TRAIN_DIR using
# tf.keras.utils.image_dataset_from_directory with labels="inferred",
# label_mode="categorical", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
# validation_split=VAL_SPLIT, subset="training", seed=SEED, and shuffle=True.

class_names = raw_train_ds.class_names
print("Class order:", class_names)

# TODO 2: Load the validation subset from TRAIN_DIR with the same loader,
# validation split, seed, image size, batch size, and categorical labels.
# Use subset="validation", shuffle=True, and class_names=class_names.

# TODO 3: Load the independent test dataset from TEST_DIR with inferred
# categorical labels, image_size=IMG_SIZE, batch_size=BATCH_SIZE,
# shuffle=False, and class_names=class_names.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE

# TODO 4: Create a tf.keras.layers.Rescaling layer that maps image pixels
# from [0, 255] to [0, 1].

# TODO 5: Use dataset.map(..., num_parallel_calls=AUTOTUNE) to apply the
# normalization layer to raw_train_ds, raw_val_ds, and raw_test_ds,
# producing train_ds, val_ds, and test_ds.

# TODO 6: Apply cache().prefetch(buffer_size=AUTOTUNE) to train_ds,
# val_ds, and test_ds.

print("Datasets are ready.")


## 7. Preview training images


In [ ]:
plt.figure(figsize=(8, 8))
for images, labels in raw_train_ds.take(1):
    for index in range(min(9, images.shape[0])):
        ax = plt.subplot(3, 3, index + 1)
        plt.imshow(images[index].numpy().astype("uint8"))
        plt.title(class_names[int(np.argmax(labels[index].numpy()))])
        plt.axis("off")

plt.tight_layout()
plt.show()


## 8. Build the CNN

The required model uses convolution, max pooling, dropout, flattening, dense hidden layers, and one softmax output unit per malware family.


In [ ]:
NUM_CLASSES = len(class_names)


def build_malware_cnn(num_classes: int) -> tf.keras.Model:
    model = tf.keras.Sequential([
        # TODO 7: Add the same CNN layers used in the solution:
        # Input(shape=(64, 64, 3)),
        # Conv2D(30, kernel_size=(3, 3), activation="relu"),
        # MaxPooling2D(pool_size=(2, 2)),
        # Conv2D(15, kernel_size=(3, 3), activation="relu"),
        # MaxPooling2D(pool_size=(2, 2)),
        # Dropout(0.25),
        # Flatten(),
        # Dense(128, activation="relu"),
        # Dropout(0.50),
        # Dense(50, activation="relu"),
        # Dense(num_classes, activation="softmax").
    ])

    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"],
    )
    return model


model = build_malware_cnn(NUM_CLASSES)
model.summary()


## 9. Train the model

Keep the independent test set out of training and model-selection decisions.


In [ ]:
EPOCHS = 10

start_time = time.time()

# TODO 8: Train model with model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS,
# ) and store the returned History object in history.

training_minutes = (time.time() - start_time) / 60
print(f"Training time: {training_minutes:.2f} minutes")


## 10. Plot training and validation results


In [ ]:
def plot_history(history: tf.keras.callbacks.History) -> None:
    metrics_history = history.history

    plt.figure(figsize=(7, 5))
    plt.plot(metrics_history.get("accuracy", []), label="training accuracy")
    plt.plot(metrics_history.get("val_accuracy", []), label="validation accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title("Training and Validation Accuracy")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(7, 5))
    plt.plot(metrics_history.get("loss", []), label="training loss")
    plt.plot(metrics_history.get("val_loss", []), label="validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training and Validation Loss")
    plt.legend()
    plt.grid(True)
    plt.show()


plot_history(history)


## 11. Evaluate validation and independent test performance

Use `studentTest200/` only for final evaluation and analysis, not for training or hyperparameter tuning.


In [ ]:
# TODO 9: Evaluate model on val_ds with model.evaluate(..., verbose=1)
# and assign the returned values to val_loss and val_acc.

# TODO 10: Evaluate model on test_ds with model.evaluate(..., verbose=1)
# and assign the returned values to test_loss and test_acc.

print(f"Validation accuracy: {val_acc * 100:.2f}%")
print(f"Student-test accuracy: {test_acc * 100:.2f}%")


## 12. Save the trained model

The required deliverable uses `.h5` for compatibility with the existing module materials. Keras may report that HDF5 is a legacy format; this warning does not prevent the required save/reload workflow.


In [ ]:
model.save(str(MODEL_PATH))
print(f"Saved model to: {MODEL_PATH}")

if SAVE_MODEL_TO_DRIVE:
    DRIVE_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(MODEL_PATH, DRIVE_MODEL_PATH)
    print(f"Copied model to Google Drive: {DRIVE_MODEL_PATH}")
else:
    print("Model saved only in /content. Set SAVE_MODEL_TO_DRIVE=True to copy it to MyDrive.")


## 13. Reload the saved `.h5` model and evaluate it again


In [ ]:
# TODO 11: Load the saved .h5 model from MODEL_PATH with
# tf.keras.models.load_model(str(MODEL_PATH), compile=False)
# and assign it to loaded_model.

loaded_model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"],
)
loaded_model.summary()

loaded_test_loss, loaded_test_acc = loaded_model.evaluate(test_ds, verbose=1)
print(f"Loaded-model student-test accuracy: {loaded_test_acc * 100:.2f}%")


## 14. Generate the classification report and confusion matrix

This section generates the evaluation outputs needed for the post-assignment questions.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

y_true: list[int] = []
y_pred: list[int] = []

# TODO 12: Loop over test_ds. For each batch:
# 1. call loaded_model.predict(batch_images, verbose=0),
# 2. convert one-hot labels to class indices with np.argmax(..., axis=1),
# 3. convert predicted probabilities to class indices with np.argmax(..., axis=1),
# 4. extend y_true and y_pred with the resulting Python lists.

report_text = classification_report(
    y_true,
    y_pred,
    target_names=class_names,
)
print(report_text)

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(range(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(range(len(class_names)), class_names)
plt.colorbar()

for row_index in range(cm.shape[0]):
    for column_index in range(cm.shape[1]):
        plt.text(
            column_index,
            row_index,
            cm[row_index, column_index],
            ha="center",
            va="center",
        )

plt.tight_layout()
plt.show()


## 15. Interpretation Guidance

Use the training and validation curves, validation accuracy, independent-test accuracy, loaded-model accuracy, classification report, and confusion matrix generated by your own run to complete the post-assignment questions in the student guide.

Reference answers and recorded-run interpretations are intentionally omitted from the student notebook.


## 16. Optional browser download

Set `DOWNLOAD_MODEL = True` only when you want Colab to download the `.h5` file through the browser.


In [ ]:
DOWNLOAD_MODEL = False

if DOWNLOAD_MODEL:
    from google.colab import files

    files.download(str(MODEL_PATH))
else:
    print("Skipping browser download.")


## 17. Optional extension tasks (bonus)

These tasks are not part of the 12 required TODOs.

1. **Controlled architecture comparison:** Change one major CNN design choice, keep the rest of the pipeline fixed, retrain, and compare validation and independent-test results with the baseline.
2. **Misclassified-sample inspection:** Use the optional helper below to display incorrectly classified test images and discuss whether visible texture similarities may explain the errors.
3. **Native Keras-format comparison:** Save the trained model as `.keras`, reload it, and verify that its independent-test accuracy is consistent with the `.h5` model.


In [ ]:
RUN_BONUS_MISCLASSIFIED = False
RUN_BONUS_KERAS_SAVE = False

if RUN_BONUS_MISCLASSIFIED:
    # Optional scaffold: display up to nine misclassified images.
    # Complete or adapt this block as needed.
    shown = 0
    plt.figure(figsize=(10, 10))

    for raw_images, batch_labels in raw_test_ds:
        normalized_images = normalization_layer(raw_images)
        probabilities = loaded_model.predict(normalized_images, verbose=0)
        true_indices = np.argmax(batch_labels.numpy(), axis=1)
        predicted_indices = np.argmax(probabilities, axis=1)

        for image_index in range(raw_images.shape[0]):
            if true_indices[image_index] == predicted_indices[image_index]:
                continue

            shown += 1
            ax = plt.subplot(3, 3, shown)
            plt.imshow(raw_images[image_index].numpy().astype("uint8"))
            plt.title(
                f"True: {class_names[true_indices[image_index]]}\n"
                f"Pred: {class_names[predicted_indices[image_index]]}"
            )
            plt.axis("off")

            if shown == 9:
                break

        if shown == 9:
            break

    plt.tight_layout()
    plt.show()

if RUN_BONUS_KERAS_SAVE:
    keras_model_path = PROJECT_DIR / "malware_cnn_model.keras"
    loaded_model.save(str(keras_model_path))
    native_model = tf.keras.models.load_model(str(keras_model_path))
    native_loss, native_acc = native_model.evaluate(test_ds, verbose=1)
    print(f"Native-format test accuracy: {native_acc * 100:.2f}%")


## Notes

- The required folders are `/content/train400` and `/content/studentTest200`.
- `/content/anomalies` is optional and is not used by the core CNN pipeline.
- The folders are downloaded from GitHub using sparse checkout; no ZIP file or extraction step is used.
- `/content` is temporary. Rerun the download and setup cells after a new Colab runtime starts.
- `studentTest200` must not be used for training or hyperparameter tuning.
- The required model is saved to `/content/malware_cnn_model.h5`.
- Set `SAVE_MODEL_TO_DRIVE = True` only when a persistent copy in `MyDrive` is desired.
